# Tier 0 Assignment 1 — Solutions: Computational Foundations

Complete solutions for all 8 problems.


## Complicated moments explained
These are common points where learners usually get stuck:
- Difference between command syntax and conceptual model (what changes state vs what only inspects).
- Interpreting statistical output: p-value alone is not enough without effect size and assumptions.
- Reproducibility: the same command can yield different outputs if input files or working directory differ.


In [ ]:
import math
import re

## Problem 1: Shell Pipeline Simulator

Filter by substring, extract a field, count occurrences, sort by count descending.

In [ ]:
def shell_pipeline(lines: list, pattern: str, field: int, top_n: int) -> list:
    # grep: keep lines containing pattern
    filtered = [line for line in lines if pattern in line]

    # cut: extract the field; skip lines that are too short
    fields = []
    for line in filtered:
        parts = line.split()
        if field < len(parts):
            fields.append(parts[field])

    # uniq -c: count occurrences
    counts = {}
    for value in fields:
        counts[value] = counts.get(value, 0) + 1

    # sort -rn | head: sort by count descending, take top_n
    sorted_counts = sorted(counts.items(), key=lambda x: x[1], reverse=True)
    return [(count, value) for value, count in sorted_counts[:top_n]]


# Tests
lines = [
    "error disk full",
    "error network timeout",
    "warning disk space",
    "error disk corrupt",
]
print(shell_pipeline(lines, pattern="error", field=1, top_n=2))
# Expected: [(2, 'disk'), (1, 'network')]

log_lines = [
    "INFO server started",
    "ERROR server crashed",
    "ERROR server crashed",
    "INFO server started",
    "ERROR client disconnected",
]
print(shell_pipeline(log_lines, pattern="ERROR", field=1, top_n=3))
# Expected: [(2, 'server'), (1, 'client')]

## Problem 2: Git Log Parser

Use a regex to find each commit block, then extract the three fields.

In [ ]:
def parse_git_log(log_text: str) -> list:
    # Each commit block: 'commit <hash>\nDate:   <date>\n\n    <message>\n'
    pattern = re.compile(
        r"commit\s+(\S+)\n"
        r"Date:\s+(.+)\n"
        r"\n"
        r"\s+(.+)",
        re.MULTILINE,
    )
    commits = []
    for match in pattern.finditer(log_text):
        full_hash, date, message = match.groups()
        commits.append({
            "hash": full_hash[:7],
            "date": date.strip(),
            "message": message.strip(),
        })
    return commits


# Tests
sample_log = """commit a1b2c3d4e5f67890abcdef1234567890abcdef12
Date:   Mon Apr 10 09:00:00 2024 +0000

    Add genome alignment module

commit deadbeef0123456789abcdef0123456789abcdef
Date:   Sun Apr 9 18:30:00 2024 +0000

    Fix off-by-one in FASTA parser

commit cafebabe1234567890abcdef1234567890abcdef
Date:   Sat Apr 8 12:15:00 2024 +0000

    Initial commit
"""

for c in parse_git_log(sample_log):
    print(c)

## Problem 3: Bash Variable Expander

Use `re.sub` with a single pattern that matches both `${VAR}` and `$VAR` forms.
The braced form is checked first by ordering alternatives in the regex.

In [ ]:
def expand_vars(template: str, variables: dict) -> str:
    # Match ${VAR} first, then $VAR (longest identifier)
    pattern = re.compile(r"\$\{([A-Za-z_][A-Za-z0-9_]*)\}|\$([A-Za-z_][A-Za-z0-9_]*)")

    def replace(match):
        # Group 1 is the braced form; group 2 is the unbraced form
        name = match.group(1) if match.group(1) is not None else match.group(2)
        return variables.get(name, "")

    return pattern.sub(replace, template)


# Tests
print(expand_vars("Hello $NAME, you are $AGE years old", {"NAME": "Alice", "AGE": "30"}))
# Expected: Hello Alice, you are 30 years old

print(expand_vars("Path: ${HOME}/data/${FILE}", {"HOME": "/usr/local", "FILE": "genome.fa"}))
# Expected: Path: /usr/local/data/genome.fa

print(expand_vars("$UNDEFINED is gone", {}))
# Expected:  is gone

print(expand_vars("${A}${B}_suffix", {"A": "chr", "B": "1"}))
# Expected: chr1_suffix

## Problem 4: File Encoding Detector

Check BOM signatures first (3-byte UTF-8 BOM before 2-byte UTF-16 BOMs), then
fall back to byte-range inspection and a UTF-8 decode attempt.

In [ ]:
def detect_encoding(data: bytes) -> str:
    if data[:3] == b"\xef\xbb\xbf":
        return "utf-8-bom"
    if data[:2] == b"\xff\xfe":
        return "utf-16-le"
    if data[:2] == b"\xfe\xff":
        return "utf-16-be"
    if all(b < 128 for b in data):
        return "ascii"
    try:
        data.decode("utf-8")
        return "utf-8"
    except UnicodeDecodeError:
        return "unknown"


# Tests
print(detect_encoding(b"Hello world"))             # ascii
print(detect_encoding(b"\xef\xbb\xbfHello"))      # utf-8-bom
print(detect_encoding(b"\xff\xfeHello"))           # utf-16-le
print(detect_encoding(b"\xfe\xffHello"))           # utf-16-be
print(detect_encoding("Héllo".encode("utf-8")))    # utf-8
print(detect_encoding(b"\x80\x81\x82"))           # unknown

## Problem 5: R Vector Statistics

Sort the data, compute mean directly, and use linear interpolation (R type=7)
for Q1, median, and Q3.

In [ ]:
def r_summary(data: list) -> dict:
    xs = sorted(data)
    n = len(xs)

    def quantile(q):
        """Linear interpolation quantile (R type=7)."""
        h = q * (n - 1)          # virtual 0-based index
        lo = int(math.floor(h))
        frac = h - lo
        if frac == 0 or lo + 1 >= n:
            return float(xs[lo])
        return xs[lo] + frac * (xs[lo + 1] - xs[lo])

    mean = sum(xs) / n

    return {
        "min":    round(float(xs[0]), 2),
        "Q1":     round(quantile(0.25), 2),
        "median": round(quantile(0.50), 2),
        "mean":   round(mean, 2),
        "Q3":     round(quantile(0.75), 2),
        "max":    round(float(xs[-1]), 2),
    }


# Tests
print(r_summary([1, 2, 3, 4, 5]))
# Expected: {'min': 1.0, 'Q1': 2.0, 'median': 3.0, 'mean': 3.0, 'Q3': 4.0, 'max': 5.0}

print(r_summary([4, 8, 15, 16, 23, 42]))
# Expected: {'min': 4.0, 'Q1': 8.75, 'median': 15.5, 'mean': 18.0, 'Q3': 22.25, 'max': 42.0}

print(r_summary([10]))
# Expected: {'min': 10.0, 'Q1': 10.0, 'median': 10.0, 'mean': 10.0, 'Q3': 10.0, 'max': 10.0}

## Problem 6: Chi-Square Goodness-of-Fit Test

Compute expected counts from proportions × total, apply χ² = Σ (O-E)²/E,
degrees of freedom = number of categories − 1.

In [ ]:
def chi_square_gof(observed: list, expected_proportions: list) -> dict:
    total = sum(observed)
    chi2 = 0.0
    for o, prop in zip(observed, expected_proportions):
        e = prop * total
        chi2 += (o - e) ** 2 / e
    return {
        "chi2": round(chi2, 4),
        "df": len(observed) - 1,
    }


# Tests
print(chi_square_gof([50, 30, 20], [0.5, 0.3, 0.2]))
# Expected: {'chi2': 0.0, 'df': 2}

print(chi_square_gof([60, 20, 20], [0.5, 0.3, 0.2]))
# Expected: {'chi2': 6.6667, 'df': 2}

print(chi_square_gof([10, 10, 10, 10], [0.25, 0.25, 0.25, 0.25]))
# Expected: {'chi2': 0.0, 'df': 3}

## Problem 7: Binomial Probability

Compute C(n, k) iteratively with integer arithmetic to avoid overflow, then
multiply by p^k * (1-p)^(n-k). The CDF sums PMF(i) for i in 0..k.

In [ ]:
def _comb(n: int, k: int) -> int:
    """Compute C(n, k) using integer arithmetic."""
    if k < 0 or k > n:
        return 0
    k = min(k, n - k)   # use smaller k for efficiency
    result = 1
    for i in range(k):
        result = result * (n - i) // (i + 1)
    return result


def binomial_pmf(n: int, k: int, p: float) -> float:
    return _comb(n, k) * (p ** k) * ((1 - p) ** (n - k))


def binomial_cdf(n: int, k: int, p: float) -> float:
    return sum(binomial_pmf(n, i, p) for i in range(k + 1))


# Tests
print(round(binomial_pmf(10, 3, 0.5), 4))   # Expected: 0.1172
print(round(binomial_cdf(10, 3, 0.5), 4))   # Expected: 0.1719
print(round(binomial_pmf(10, 5, 0.5), 4))   # Expected: 0.2461
print(round(binomial_cdf(10, 5, 0.5), 4))   # Expected: 0.623
print(round(binomial_pmf(20, 10, 0.3), 6))  # Expected: 0.030782
print(round(binomial_cdf(20, 10, 0.3), 6))  # Expected: 0.982791

## Problem 8: Independent T-Test (Welch's)

Compute sample means and sample variances (n-1 denominator), then apply
Welch's t-statistic and Welch-Satterthwaite degrees of freedom formulas.

In [ ]:
def welch_t_test(group1: list, group2: list) -> dict:
    n1, n2 = len(group1), len(group2)

    mean1 = sum(group1) / n1
    mean2 = sum(group2) / n2

    # Sample variance (Bessel's correction: divide by n-1)
    var1 = sum((x - mean1) ** 2 for x in group1) / (n1 - 1)
    var2 = sum((x - mean2) ** 2 for x in group2) / (n2 - 1)

    se1 = var1 / n1   # variance of mean for group 1
    se2 = var2 / n2   # variance of mean for group 2

    t_stat = (mean1 - mean2) / math.sqrt(se1 + se2)

    # Welch-Satterthwaite degrees of freedom
    df = (se1 + se2) ** 2 / (se1 ** 2 / (n1 - 1) + se2 ** 2 / (n2 - 1))

    return {
        "t_stat": round(t_stat, 4),
        "df": round(df, 4),
    }


# Tests
print(welch_t_test([2.1, 2.5, 2.3, 2.7], [1.8, 1.6, 2.0, 1.9]))
# Expected: {'t_stat': 3.6515, 'df': 5.9965}  (approximately)

print(welch_t_test([5.0, 5.0, 5.0], [5.0, 5.0, 5.0]))
# Both groups identical — t_stat should be 0 (will raise ZeroDivisionError if var=0)
# Note: mathematically undefined when both variances are 0; handled separately in practice

print(welch_t_test([1, 2, 3, 4, 5], [6, 7, 8, 9, 10]))
# Expected: large negative t_stat, df close to 8.0

---

## Key Patterns

1. **Shell Pipeline**: chain filter → extract → count → sort as separate list comprehension/dict steps
2. **Git Log Parser**: `re.compile` with `re.MULTILINE` and `finditer` is cleaner than splitting on blank lines
3. **Variable Expander**: one regex with two alternating groups handles both `${VAR}` and `$VAR`; group 1 wins when braced form matches
4. **Encoding Detector**: check the 3-byte UTF-8 BOM before the 2-byte UTF-16 BOMs; fall back to a decode attempt
5. **R Quantiles**: `h = q * (n-1)` gives the 0-based virtual index; linear interpolation between `xs[floor(h)]` and `xs[ceil(h)]`
6. **Chi-Square**: Σ (O-E)²/E; expected = proportion × total; df = categories − 1
7. **Binomial**: compute `C(n,k)` iteratively with integer division to stay exact; PMF and CDF are then trivial
8. **Welch's t-test**: sample variance (n-1), Satterthwaite df formula uses (se1+se2)² / (se1²/(n1-1) + se2²/(n2-1))
